# TeleRAG-Agent: Pipeline Verification on Kaggle

This notebook tests the three core modules on Kaggle T4 GPU:
- **Cell 1:** Install dependencies
- **Cell 2:** Test model loader (Task 1)
- **Cell 3:** Test inference module (Task 2)
- **Cell 4:** Test full RAG pipeline end-to-end (Task 3)

Upload the following files to Kaggle alongside this notebook:
- `kaggle_eval_golden_final.json` (from data/processed/)

In [ ]:
# Cell 1: Setup
!pip install unsloth transformers accelerate bitsandbytes peft sentence-transformers qdrant-client -q

import torch
import re
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# Cell 2: Test Model Loader (Task 1)
# -----------------------------------
# Tests: model loads in 4-bit, tokenizer has pad token, singleton works

from kaggle_secrets import UserSecretsClient
try:
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except:
    hf_token = ''

BASE_MODEL = 'AliMaatouk/LLama-3-8B-Tele-it'
LORA_REPO  = None  # Set to your HF repo e.g. 'chaitanyakadupukutla/TeleRAG-LoRA' to test with LoRA

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
print(f'[PASS] Tokenizer loaded. Pad token: {tokenizer.pad_token!r}')

print('Loading model in 4-bit...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    token=hf_token,
)

if LORA_REPO:
    from peft import PeftModel
    print(f'Merging LoRA from {LORA_REPO}...')
    model = PeftModel.from_pretrained(model, LORA_REPO, token=hf_token)
    print('[PASS] LoRA merged')

model.eval()
print(f'[PASS] Model loaded: {type(model).__name__}')
print(f'[PASS] Device: {next(model.parameters()).device}')

In [ ]:
# Cell 3: Test Inference Module (Task 2)
# ---------------------------------------
# Tests: prompt builds correctly, model generates output, letter extraction works

def build_mcq_prompt(question, options, context=''):
    """Exact same function as src/models/inference.py"""
    opts_str = '\n'.join(f'{chr(65+i)}) {o}' for i, o in enumerate(options))
    q_block = f'Question: {question}\n\nOptions:\n{opts_str}'
    if context and len(context.strip()) > 100:
        q_block = f'Relevant information:\n{context[:3000]}\n\n' + q_block
    return f'### Question:\n{q_block}\n\n### Answer:\n'

def extract_letter(response, num_options=5):
    max_letter = chr(64 + num_options)
    m = re.search(r'The answer is:\s*([A-Z])', response, re.IGNORECASE)
    if m:
        letter = m.group(1).upper()
        if 'A' <= letter <= max_letter:
            return letter
    m = re.search(rf'\b([A-{max_letter}])\b', response[:50], re.IGNORECASE)
    return m.group(1).upper() if m else None

# Test prompt format
test_q = 'What is DRX in 5G NR?'
test_opts = ['Discontinuous Reception', 'Data Relay Extension', 'Dynamic Resource eXchange', 'None of the above']
prompt = build_mcq_prompt(test_q, test_opts)
assert '### Question:' in prompt, 'Missing prompt header'
assert '### Answer:' in prompt, 'Missing answer trigger'
assert 'A) Discontinuous Reception' in prompt
print('[PASS] MCQ prompt format is correct')
print(f'Prompt:\n{prompt}')

# Test generation
import time
inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=4096).to(model.device)
t0 = time.perf_counter()
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
elapsed = (time.perf_counter() - t0) * 1000
response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
print(f'\n[PASS] Model generated in {elapsed:.0f}ms: "{response}"')

# Test extraction
letter = extract_letter(response, num_options=4)
print(f'[PASS] Extracted letter: {letter}')

In [ ]:
# Cell 4: Test Full Pipeline on Golden Dataset (Task 3)
# -------------------------------------------------------
# Runs a mini-evaluation on 10 questions from the golden dataset
# to verify retrieval -> generation -> accuracy end-to-end

import json

# Load 10 golden questions
with open('/kaggle/input/telerag-eval/kaggle_eval_golden_final.json') as f:
    eval_data = json.load(f)[:10]  # Just 10 for quick verification
print(f'Loaded {len(eval_data)} golden questions')

correct = 0
results = []

for i, item in enumerate(eval_data):
    # Build prompt (with RAG context already in the golden dataset)
    prompt = build_mcq_prompt(
        question=item['question'],
        options=item['options'],
        context=item.get('context', ''),
    )

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=3800).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=15,
            do_sample=False,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    predicted = extract_letter(response, num_options=len(item['options']))
    actual = item['answer'].strip().upper()
    is_correct = (predicted == actual)
    if is_correct:
        correct += 1

    results.append({'q': item['question'][:60], 'pred': predicted, 'actual': actual, 'ok': is_correct, 'raw': response})
    print(f'Q{i+1}: Pred={predicted} Actual={actual} {"✓" if is_correct else "✗"}  raw="{response}"')

acc = correct / len(eval_data) * 100
print(f'\n=== MINI EVALUATION ===')
print(f'Correct: {correct}/{len(eval_data)}')
print(f'Accuracy: {acc:.1f}%')
print(f'[PASS] Pipeline runs end-to-end' if correct > 0 else '[WARN] 0 correct — check prompt format')